# Federated Deep Learning for Financial Fraud Detection

This notebook demonstrates how to implement federated learning for financial fraud detection using NVIDIA FLARE (NVFlare). The notebook shows a complete workflow for training a deep learning model across multiple clients while maintaining data privacy.

## 1. Define Recipe

By default, we can use a small randomly generated dataset to illustrate the workflow. If the full sythentic dataset should be used, please provide the filepath with the `--dataset` train argument.

In [ ]:
from src.dl.model import SimpleNetwork

from nvflare.app_opt.tf.recipes.fedavg import FedAvgRecipe
from nvflare.recipe.sim_env import SimEnv
from nvflare.recipe.prod_env import ProdEnv

recipe = FedAvgRecipe(
    name="hello-tf-mlflow",
    min_clients=1,
    num_rounds=10,
    initial_model=SimpleNetwork(num_classes=2),
    train_script="src/dl/client.py",
    train_args="--dataset None", #"/workspace/dataset/paysim1/PS_20174392719_1491204439457_log.csv"
)

### Add Experiment Tracking

In [ ]:
# Add experiment tracking
from nvflare.recipe.utils import add_experiment_tracking

mlflow_config = {
    "tracking_uri": "https://rrayrz6j-nvflmlflowserver.xenon.lepton.run",
    "kw_args": {
        "experiment_name": "nvflare-poc-lepton-fl-experiment",
        "run_name": "nvflare-fedavgrecipe-with-mlflow-01",
        "experiment_tags": {"mlflow.note.content": "## **NVFlare FedAvg experiment with MLflow**"},
        "run_tags": {"mlflow.note.content": "## Federated Experiment tracking with MLflow.\n"},
    },
    "artifact_location": "artifacts",
    "events": ["fed.analytix_log_stats"],
}
add_experiment_tracking(recipe, tracking_type="mlflow", tracking_config=mlflow_config)

### Optionall export

In [ ]:
recipe.export("job_configs")

## 2. Run in Simulation Environment

In [ ]:
#env = SimEnv(num_clients=1)
#recipe.execute(env=env)

## 3. Run in Production Environment

In [ ]:
env = ProdEnv(startup_kit_location="/scratch/hroth/Code/JPM/admin", username="admin@nvidia.com")
run = recipe.execute(env=env)

### Get Status

In [ ]:
print("Job Status is:", run.get_status())

### Get Results

In [ ]:
print("Result can be found in:", run.get_result())

.

### Visualization of Training Progress

In the meantime, you can check the progress via mlflow: [https://rrayrz6j-nvflmlflowserver.xenon.lepton.run](https://rrayrz6j-nvflmlflowserver.xenon.lepton.run)

The mflow metrics should show the accuracy progressing at each round:

![mlflow model metrics](./figs/mlflow.png)

### Optionally Abort Job
If you'd like to stop the job before completion for any reason, you can run:

In [ ]:
run.abort()